# Figure 1 — Sampling Density Mismatch Biases EOT

Reproduces **Figure 1** from *Density-Reweighted Entropic Optimal Transport*.

**Setup:** Two datasets on 1-D manifolds embedded in R²:
- Dataset X (line):  y = x, with x ~ piecewise-uniform, 9:1 density ratio (left-heavy)
- Dataset Y (curve): y = 2 + x + 0.5x², with x ~ piecewise-uniform, 1:9 ratio (right-heavy)

**Methods compared:** Standard EOT (θ=0) vs DR-EOT (θ=1) using ground-truth densities.

**Parameters:** n=3000, m=2000, ε=5×10⁻².

In [ ]:
import sys
sys.path.insert(0, '..')   # so `dreot` is importable from the repo root

import numpy as np
from sklearn.metrics import pairwise_distances
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from dreot import sinkhorn_eot, sinkhorn_dreot

## 1. Data Generation

In [ ]:
class PiecewiseUniform:
    """Piecewise-uniform distribution on [0,1] with a single break point."""
    def __init__(self, break_point=0.5, weights=(1, 1)):
        self.bp = break_point
        prob = break_point * weights[0] + (1 - break_point) * weights[1]
        self.w = [weights[0] / prob, weights[1] / prob]
        self.coin = break_point * weights[0] / prob

    def sample(self, n):
        c = np.random.uniform(size=n)
        n1 = int(np.sum(c <= self.coin))
        x = np.concatenate([
            np.random.uniform(0, self.bp, n1),
            np.random.uniform(self.bp, 1, n - n1),
        ])
        return np.sort(x).reshape(-1, 1)

    def pdf(self, x):
        return np.where(np.asarray(x) < self.bp, self.w[0], self.w[1])


# ── Paper parameters ──────────────────────────────────────────────────────────
np.random.seed(42)
m, n    = 2000, 3000   # X has m points, Y has n points
c_curve = 0.5          # quadratic coefficient of the curve y = 2 + x + c*x^2
eps     = 5e-2         # entropic regularization ε

samplerX = PiecewiseUniform(0.5, (9, 1))  # heavy on [0, 0.5]
samplerY = PiecewiseUniform(0.5, (1, 9))  # heavy on [0.5, 1]

x_X = samplerX.sample(m)   # (m, 1) — first coordinate of X
x_Y = samplerY.sample(n)   # (n, 1) — first coordinate of Y

X = np.hstack([x_X, x_X])                            # line: y = x
Y = np.hstack([x_Y, 2 + x_Y + c_curve * x_Y**2])    # curve

# True arc-length densities  (pdf divided by |gamma'|)
mu = samplerX.pdf(x_X) / np.sqrt(2)                             # (m,)
nu = samplerY.pdf(x_Y) / np.sqrt(1 + (1 + 2*c_curve*x_Y)**2)   # (n,)

print(f"X shape: {X.shape}  Y shape: {Y.shape}")
print(f"ε = {eps}")

## 2. Compute Transport Plans

In [ ]:
dist = pairwise_distances(X, Y, metric="sqeuclidean")

# Standard EOT — uniform marginals
row_sum = np.ones((m, 1)) * n
col_sum = np.ones((n, 1)) * m
row_s, col_s = sinkhorn_eot(
    dist, eps, row_sum, col_sum,
    delta=1e-6, max_iter=3000, check_freq=10, raise_on_bad_convergence=False,
)
W_eot = row_s * np.exp(-dist / eps) * col_s.T

# DR-EOT — θ=1, true densities
row_s_dr, col_s_dr = sinkhorn_dreot(
    dist, eps,
    mu.reshape(-1, 1), nu.reshape(-1, 1),
    alpha=1,
    delta=1e-6, max_iter=3000, check_freq=10, raise_on_bad_convergence=False,
)
W_dreot = row_s_dr * np.exp(-dist / eps) * col_s_dr.T

print(f"W_eot  shape: {W_eot.shape}")
print(f"W_dreot shape: {W_dreot.shape}")

## 3. Figure 1 — Maximum-coupling correspondences

In [ ]:
# ── Publication style ──────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["TeX Gyre Termes", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 0, "ytick.labelsize": 0,
    "axes.linewidth": 0.6, "axes.grid": False,
    "figure.dpi": 150, "savefig.dpi": 600,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

ARROW_COLOR = "gray"
vmin = min(mu.min(), nu.min())
vmax = max(mu.max(), nu.max())

fig, axes = plt.subplots(1, 2, figsize=(8, 2.5), sharey=False)

for ax, W, title, panel in zip(
        axes,
        [W_eot, W_dreot],
        ["EOT", "Our Approach (DR-EOT, θ=1)"],
        ["(a)", "(b)"]):

    sc = ax.scatter(X[:, 0], X[:, 1], c=mu, cmap="viridis",
                    s=5, vmin=vmin, vmax=vmax, linewidths=0, rasterized=True, zorder=3)
    ax.scatter(Y[:, 0], Y[:, 1], c=nu, cmap="viridis",
               s=5, vmin=vmin, vmax=vmax, linewidths=0, rasterized=True, zorder=3)

    # Maximum-coupling arrows (~100 arrows for clarity)
    max_idx = np.argmax(W, axis=1)
    step = max(1, m // 100)
    for i in range(0, m, step):
        j = max_idx[i]
        ax.annotate("",
                    xy=(Y[j, 0], Y[j, 1]),
                    xytext=(X[i, 0], X[i, 1]),
                    arrowprops=dict(arrowstyle="-|>", color=ARROW_COLOR,
                                   lw=0.5, alpha=0.4, mutation_scale=4),
                    zorder=2)

    ax.text(-0.08, 1.07, panel, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")
    ax.set_title(title, pad=5)
    ax.set_xticks([]); ax.set_yticks([])

divider = make_axes_locatable(axes[1])
cax = divider.append_axes("right", size="4%", pad=0.05)
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label("Sampling Density", labelpad=6, fontsize=8)
cbar.set_ticks([])

plt.tight_layout(w_pad=1.2)
plt.savefig("fig1_demo_mismatch.pdf")
plt.show()
print("Saved fig1_demo_mismatch.pdf")